# 2주차 라이브 코딩: nn.Module MLP 정의 → 역전파 → 학습

> **수업 시연용 노트북** — ch2A 이론 강의 후 교수가 시연하거나, 학생이 시연 영상을 보며 따라가는 용도

### 목표
1. `nn.Module`로 간단한 2층 MLP를 정의한다
2. 한 번의 역전파 과정을 단계별로 추적한다
3. 전체 학습 루프로 손실이 감소하는 것을 확인한다

---
## Stage 1: nn.Module로 모델 정의하기

PyTorch에서 모든 신경망은 `nn.Module`을 상속받아 만든다.
- `__init__`: 모델의 구조(층)를 정의
- `forward`: 데이터가 모델을 통과하는 방식을 정의

In [ ]:
import torch
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력 → 은닉
        self.relu = nn.ReLU()                           # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size)  # 은닉 → 출력

    def forward(self, x):
        x = self.fc1(x)      # 첫 번째 완전 연결층
        x = self.relu(x)     # ReLU 활성화
        x = self.fc2(x)      # 두 번째 완전 연결층
        return x

In [ ]:
# 모델 생성
model = SimpleMLP(input_size=10, hidden_size=32, output_size=2)

# 모델 구조 확인
print(model)

# 파라미터 개수 계산
# fc1: 10×32 + 32 = 352
# fc2: 32×2 + 2 = 66
# 총: 418개
total_params = sum(p.numel() for p in model.parameters())
print(f"\n모델 파라미터 수: {total_params}")

# 각 층별 파라미터
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape} ({param.numel()}개)")

---
## Stage 2: 손실/옵티마이저 설정 + 한 번의 역전파 추적

학습의 5단계:
1. `optimizer.zero_grad()` — 이전 그래디언트 초기화
2. `output = model(X)` — 순전파
3. `loss = criterion(output, y)` — 손실 계산
4. `loss.backward()` — 역전파 (그래디언트 계산)
5. `optimizer.step()` — 파라미터 업데이트

In [ ]:
# 손실 함수와 옵티마이저 설정
criterion = nn.CrossEntropyLoss()        # 분류 손실 함수
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 더미 데이터 생성
X = torch.randn(4, 10)   # 배치 크기 4, 입력 차원 10
y = torch.tensor([0, 1, 0, 1])  # 4개 샘플의 정답

print(f"입력 크기: {X.shape}")
print(f"정답: {y.tolist()}")

In [ ]:
# 순전파
output = model(X)              # (4, 2) — 4개 샘플, 2개 클래스
loss = criterion(output, y)
print(f"순전파 결과: {output.shape}")
print(f"손실값: {loss.item():.4f}")

In [ ]:
# 역전파 (자동 미분)
optimizer.zero_grad()       # 이전 그래디언트 초기화
loss.backward()             # 역전파: 모든 파라미터의 그래디언트 계산

# 그래디언트 확인
print("각 파라미터의 그래디언트:")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name}: shape={param.grad.shape}, norm={param.grad.norm():.4f}")

In [ ]:
# 파라미터 업데이트
optimizer.step()           # w = w - lr × ∂L/∂w

# 업데이트 후 손실 (작아져야 함)
output_after = model(X)
loss_after = criterion(output_after, y)
print(f"업데이트 전 손실: {loss.item():.4f}")
print(f"업데이트 후 손실: {loss_after.item():.4f}")
print(f"손실 감소: {(loss.item() - loss_after.item()):.6f}")
print(f"\n→ 한 번의 역전파로 손실이 줄었다!")

---
## Stage 3: 전체 학습 루프

위의 5단계를 **여러 에폭** 동안 반복하면 학습이 진행된다.

```
for epoch in range(num_epochs):
    for batch_x, batch_y in train_loader:
        output = model(batch_x)          # 순전파
        loss = criterion(output, batch_y) # 손실 계산
        optimizer.zero_grad()             # 그래디언트 초기화
        loss.backward()                   # 역전파
        optimizer.step()                  # 파라미터 업데이트
```

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# 더미 데이터셋 (100개 샘플)
torch.manual_seed(42)
X_train = torch.randn(100, 10)
y_train = torch.randint(0, 2, (100,))
dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(dataset, batch_size=16)

# 새 모델 초기화
model = SimpleMLP(input_size=10, hidden_size=32, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"데이터: {len(dataset)}개 샘플, 배치 크기: 16, 배치 수: {len(train_loader)}")

In [ ]:
# 학습 루프
num_epochs = 10
losses = []

for epoch in range(num_epochs):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        output = model(batch_x)          # 순전파
        loss = criterion(output, batch_y) # 손실 계산

        optimizer.zero_grad()             # 그래디언트 초기화
        loss.backward()                   # 역전파
        optimizer.step()                  # 파라미터 업데이트

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1:2d}/{num_epochs}: Loss = {avg_loss:.4f}")

print(f"\n→ 손실이 {losses[0]:.4f}에서 {losses[-1]:.4f}로 감소 — 학습이 진행되고 있다!")

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, num_epochs + 1), losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ 매 에폭마다 손실이 감소하는 것을 확인할 수 있다")

---
## 핵심 정리

| 개념 | 핵심 |
|------|------|
| `nn.Module` | 모든 PyTorch 모델의 부모 클래스. `__init__`에서 층 정의, `forward`에서 순전파 정의 |
| `CrossEntropyLoss` | 분류 문제의 표준 손실 함수. 내부적으로 softmax 포함 |
| `zero_grad()` | 이전 배치의 그래디언트를 초기화. 안 하면 그래디언트가 누적됨 |
| `backward()` | 역전파: 모든 파라미터의 ∂Loss/∂w를 한 번에 계산 |
| `step()` | 파라미터 업데이트: w = w - lr × ∂Loss/∂w |

### B회차 과제 미리보기

B회차에서는 이 패턴을 **한국어 영화 리뷰 감성 분류**에 적용한다:
- Bag-of-Words로 텍스트를 벡터로 변환
- MLP로 긍정/부정 분류
- Accuracy, Precision, Recall, F1 평가

참고 코드: `practice/chapter2/code/2-4-텍스트분류.py`